In [1]:
import polars as pl

train = pl.read_csv('/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv')

print(train.head(2))

shape: (2, 3)
┌──────────┬─────────────────────────────────┬──────────┐
│ id       ┆ prompt                          ┆ answer   │
│ ---      ┆ ---                             ┆ ---      │
│ str      ┆ str                             ┆ str      │
╞══════════╪═════════════════════════════════╪══════════╡
│ 00066667 ┆ In Alice's Wonderland, a secre… ┆ 10010111 │
│ 000b53cf ┆ In Alice's Wonderland, a secre… ┆ 01000011 │
└──────────┴─────────────────────────────────┴──────────┘


In [2]:
import kagglehub

# Download latest version
path = kagglehub.model_download("kienngx/nemotron-nano-30b-trained/triton/tinker-adapter")

print("Path to model files:", path)

Path to model files: /kaggle/input/models/kienngx/nemotron-nano-30b-trained/triton/tinker-adapter/1


In [3]:
import os
import re
import glob
import json
import shutil
import zipfile
import torch
import kagglehub
import polars as pl

from safetensors import safe_open
from safetensors.torch import save_file

# =========================
# CONFIG
# =========================
VARIANCE_RETAINED = 0.98
MIN_RANK_RATIO = 0.50
USE_FLOAT64_SVD = True

WORK_DIR = "/kaggle/working"
ORIG_ADAPTER = "adapter_model.safetensors"
NEW_ADAPTER = "converted_adapter_model.safetensors"

# =========================
# LOAD DATA (sanity only)
# =========================
train = pl.read_csv('/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv')
print("Train shape:", train.shape)

# =========================
# FIND ADAPTER
# =========================
adapter_dirs = [
    "/kaggle/input/datasets/assiaben/adapter-v32-epoch-3/adapter",
    "/kaggle/input/adapter-v32-epoch-3/adapter",
    "/kaggle/input/adapter",
]

ADAPTER_PATH = None

for p in adapter_dirs:
    if os.path.exists(os.path.join(p, "adapter_config.json")) and \
       os.path.exists(os.path.join(p, ORIG_ADAPTER)):
        ADAPTER_PATH = p
        break

if ADAPTER_PATH is None:
    for root, _, files in os.walk("/kaggle/input"):
        if "adapter_config.json" in files and ORIG_ADAPTER in files:
            ADAPTER_PATH = root
            break

if ADAPTER_PATH is None:
    raise FileNotFoundError("Adapter not found")

print("Adapter found:", ADAPTER_PATH)

# =========================
# COPY TO WORK DIR
# =========================
config_path = os.path.join(WORK_DIR, "adapter_config.json")
adapter_path = os.path.join(WORK_DIR, ORIG_ADAPTER)

shutil.copy2(os.path.join(ADAPTER_PATH, "adapter_config.json"), config_path)
shutil.copy2(os.path.join(ADAPTER_PATH, ORIG_ADAPTER), adapter_path)

# fix config
with open(config_path, "r") as f:
    cfg = json.load(f)

cfg.update({
    "base_model_name_or_path": "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16",
    "inference_mode": True,
    "lora_dropout": 0.0,
})

with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

print("Config updated")

# =========================
# LOAD BASE MODEL SHAPES
# =========================
MODEL_PATH = kagglehub.model_download(
    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
)

model_files = glob.glob(f"{MODEL_PATH}/*.safetensors")

model_shapes = {}

for mf in model_files:
    with safe_open(mf, framework="pt", device="cpu") as f:
        for k in f.keys():
            model_shapes[k] = tuple(f.get_slice(k).get_shape())

print("Model keys:", len(model_shapes))

# =========================
# LOAD ADAPTER
# =========================
adapter_tensors = {}

with safe_open(adapter_path, framework="pt", device="cpu") as f:
    for k in f.keys():
        adapter_tensors[k] = f.get_tensor(k)

print("Adapter tensors:", len(adapter_tensors))

# =========================
# BUILD BASE KEYS
# =========================
base_keys = set()

for k in adapter_tensors:
    base = re.sub(r"\.lora_[AB]\.weight$", "", k)
    base_keys.add(base)

print("Base keys:", len(base_keys))

# =========================
# OUTPUT TENSORS
# =========================
tensors = {}

def rename_key(k):
    return k.replace(
        "base_model.model.model",
        "base_model.model.backbone"
    )

print("Processing adapters...")

for base in sorted(base_keys):

    A = adapter_tensors.get(f"{base}.lora_A.weight")
    B = adapter_tensors.get(f"{base}.lora_B.weight")

    if A is None or B is None:
        continue

    base_renamed = rename_key(base)

    # skip empty weird tensors
    if A.numel() == 0 or B.numel() == 0:
        continue

    tensors[f"{base_renamed}.lora_A.weight"] = A.contiguous()
    tensors[f"{base_renamed}.lora_B.weight"] = B.contiguous()

# =========================
# SAVE ADAPTER
# =========================
out_path = os.path.join(WORK_DIR, NEW_ADAPTER)

save_file(tensors, out_path)

print("Saved adapter:", len(tensors))

# =========================
# ZIP SUBMISSION
# =========================
zip_path = os.path.join(WORK_DIR, "submission.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:

    z.write(config_path, "adapter_config.json")
    z.write(out_path, "adapter_model.safetensors")

print("ZIP created")

# =========================
# FINAL CHECK
# =========================
size_mb = os.path.getsize(zip_path) / 1024 / 1024
print("Final size MB:", round(size_mb, 2))

Train shape: (9500, 3)
Adapter found: /kaggle/input/models/kienngx/nemotron-nano-30b-trained/triton/tinker-adapter/1
Config updated
Model keys: 6243
Adapter tensors: 12010
Base keys: 6005
Processing adapters...
Saved adapter: 12010
ZIP created
Final size MB: 3118.8


In [4]:
ls -lh /kaggle/working/submission.zip

-rw-r--r-- 1 root root 3.1G Jun 12 11:24 /kaggle/working/submission.zip
